In [9]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
riders = pd.read_csv("../dataset/processed/riders_features.csv")

print(riders.shape)

riders.head()

(45493, 33)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Day_of_Week,Month,Weekend,Peak_Period,Trip_Distance_km,Traffic_Score,Weather_Score,Vehicle_Score,Rider_Experience,Workload
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,...,Saturday,2,1,Dinner,10.280582,4,4,4,151.2,3.0
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,...,Sunday,2,1,Lunch,6.242319,3,5,4,98.7,1.0
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,...,Friday,3,0,Normal,13.787860,2,6,3,108.1,1.0
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,...,Sunday,2,1,Breakfast,2.930258,1,6,4,146.2,0.0
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,...,Monday,2,0,Dinner,19.396618,4,4,3,112.8,1.0


In [11]:
riders["weather_norm"] = (
    riders["Weather_Score"] -
    riders["Weather_Score"].min()
) / (
    riders["Weather_Score"].max() -
    riders["Weather_Score"].min()
)

In [18]:
# Traffic normalization
riders["traffic_norm"] = (
    riders["Traffic_Score"] - riders["Traffic_Score"].min()
) / (
    riders["Traffic_Score"].max() - riders["Traffic_Score"].min()
)

print(riders["traffic_norm"].head())

0    1.000000
1    0.666667
2    0.333333
3    0.000000
4    1.000000
Name: traffic_norm, dtype: float64


In [12]:
riders["distance_norm"] = (
    riders["Trip_Distance_km"] -
    riders["Trip_Distance_km"].min()
) / (
    riders["Trip_Distance_km"].max() -
    riders["Trip_Distance_km"].min()
)

In [13]:
riders["workload_norm"] = (
    riders["Workload"] -
    riders["Workload"].min()
) / (
    riders["Workload"].max() -
    riders["Workload"].min()
)

In [15]:
print(riders.columns.tolist())

['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken (min)', 'Order_Time', 'Order_Hour', 'Pickup_Hour', 'Day_of_Week', 'Month', 'Weekend', 'Peak_Period', 'Trip_Distance_km', 'Traffic_Score', 'Weather_Score', 'Vehicle_Score', 'Rider_Experience', 'Workload', 'weather_norm', 'distance_norm', 'workload_norm']


In [16]:
print(riders.head())

       ID Delivery_person_ID  Delivery_person_Age  Delivery_person_Ratings  \
0  0xcdcd      DEHRES17DEL01                 36.0                      4.2   
1  0xd987      KOCRES16DEL01                 21.0                      4.7   
2  0x2784     PUNERES13DEL03                 23.0                      4.7   
3  0xc8b6     LUDHRES15DEL02                 34.0                      4.3   
4  0xdb64      KNPRES14DEL02                 24.0                      4.7   

   Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
0            30.327968             78.046106                   30.397968   
1            10.003064             76.307589                   10.043064   
2            18.562450             73.916619                   18.652450   
3            30.899584             75.809346                   30.919584   
4            26.463504             80.372929                   26.593504   

   Delivery_location_longitude  Order_Date Time_Orderd  ... Peak_Period  \

In [19]:
print(riders[
    ["traffic_norm", "weather_norm", "distance_norm", "workload_norm"]
].head())

   traffic_norm  weather_norm  distance_norm  workload_norm
0      1.000000           0.6       0.451975       1.000000
1      0.666667           0.8       0.244932       0.333333
2      0.333333           1.0       0.631795       0.333333
3      0.000000           1.0       0.075121       0.000000
4      1.000000           0.6       0.919358       0.333333


In [20]:
vehicle_norm = (
    riders["Vehicle_Score"] - riders["Vehicle_Score"].min()
) / (
    riders["Vehicle_Score"].max() - riders["Vehicle_Score"].min()
)

riders["FM_Score"] = (
    0.60 * riders["distance_norm"] +
    0.25 * riders["traffic_norm"] +
    0.15 * (1 - vehicle_norm)
)

In [22]:
festival = (riders["Festival"] == "Yes").astype(int)

# Adjust if your Peak_Period values are different
peak = riders["Peak_Period"].isin(["Lunch", "Dinner"]).astype(int)

riders["WT_Score"] = (
    0.50 * riders["weather_norm"] +
    0.30 * peak +
    0.20 * festival
)

In [23]:
riders["LM_Score"] = (
    0.55 * riders["distance_norm"] +
    0.30 * riders["traffic_norm"] +
    0.15 * riders["weather_norm"]
)

In [26]:
festival = (riders["Festival"] == "Yes").astype(int)

riders["O2A_Score"] = (
    0.45 * riders["traffic_norm"] +
    0.40 * riders["workload_norm"] +
    0.15 * festival
)

In [27]:
vehicle_norm = (
    riders["Vehicle_Score"] - riders["Vehicle_Score"].min()
) / (
    riders["Vehicle_Score"].max() - riders["Vehicle_Score"].min()
)

riders["FM_Score"] = (
    0.60 * riders["distance_norm"] +
    0.25 * riders["traffic_norm"] +
    0.15 * (1 - vehicle_norm)
)

In [28]:
peak = riders["Peak_Period"].isin(["Lunch", "Dinner"]).astype(int)

festival = (riders["Festival"] == "Yes").astype(int)

riders["WT_Score"] = (
    0.50 * riders["weather_norm"] +
    0.30 * peak +
    0.20 * festival
)

In [29]:
riders["LM_Score"] = (
    0.55 * riders["distance_norm"] +
    0.30 * riders["traffic_norm"] +
    0.15 * riders["weather_norm"]
)

In [30]:
print(riders.columns.tolist())

['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken (min)', 'Order_Time', 'Order_Hour', 'Pickup_Hour', 'Day_of_Week', 'Month', 'Weekend', 'Peak_Period', 'Trip_Distance_km', 'Traffic_Score', 'Weather_Score', 'Vehicle_Score', 'Rider_Experience', 'Workload', 'weather_norm', 'distance_norm', 'workload_norm', 'traffic_norm', 'FM_Score', 'WT_Score', 'LM_Score', 'O2A_Score']


In [32]:
score_sum = (
    riders["O2A_Score"] +
    riders["FM_Score"] +
    riders["WT_Score"] +
    riders["LM_Score"]
)
score_sum

0        2.609771
1        1.901671
2        1.893231
3        0.736389
4        2.930595
           ...   
45488    0.928128
45489    2.255978
45490    0.368211
45491    1.511086
45492    1.532203
Length: 45493, dtype: float64

In [33]:
score_sum = (
    riders["O2A_Score"] +
    riders["FM_Score"] +
    riders["WT_Score"] +
    riders["LM_Score"]
)

print(score_sum.head())

0    2.609771
1    1.901671
2    1.893231
3    0.736389
4    2.930595
dtype: float64


In [34]:
print((score_sum == 0).sum())

0


In [35]:
riders["O2A_Ratio"] = riders["O2A_Score"] / score_sum
riders["FM_Ratio"]  = riders["FM_Score"]  / score_sum
riders["WT_Ratio"]  = riders["WT_Score"]  / score_sum
riders["LM_Ratio"]  = riders["LM_Score"]  / score_sum

In [36]:
ratio_sum = (
    riders["O2A_Ratio"] +
    riders["FM_Ratio"] +
    riders["WT_Ratio"] +
    riders["LM_Ratio"]
)

print("Maximum Error:", np.abs(ratio_sum - 1).max())

Maximum Error: 4.440892098500626e-16


In [37]:
eta = riders["Time_taken (min)"]

riders["O2A_Target"] = eta * riders["O2A_Ratio"]
riders["FM_Target"]  = eta * riders["FM_Ratio"]
riders["WT_Target"]  = eta * riders["WT_Ratio"]
riders["LM_Target"]  = eta * riders["LM_Ratio"]

In [38]:
check = (
    riders["O2A_Target"] +
    riders["FM_Target"] +
    riders["WT_Target"] +
    riders["LM_Target"]
)

print("Maximum ETA Error:", np.abs(check - eta).max())

Maximum ETA Error: 1.4210854715202004e-14


In [39]:
cols = [
    "O2A_Target",
    "FM_Target",
    "WT_Target",
    "LM_Target"
]

print(riders[cols].describe().round(2))

       O2A_Target  FM_Target  WT_Target  LM_Target
count    45493.00   45493.00   45493.00   45493.00
mean         5.17       6.44       7.38       7.31
std          3.69       3.37       3.94       3.10
min          0.00       0.00       0.00       0.00
25%          2.55       4.15       4.77       5.00
50%          4.78       6.01       7.27       6.97
75%          7.40       8.58       9.78       9.31
max         34.72      23.16      26.43      19.61


In [40]:
riders.to_csv(
    "../dataset/processed/riders_multistage_v2.csv",
    index=False
)